# 04_Modeling

Playground Series S6E7 — Predicting Student Health Risk

Goal: train and compare a handful of baseline models using cross-validation,
pick the best one, and export a **checkpoint submission** — just to confirm
the CV score roughly matches the public leaderboard score (a sanity check,
not the final tuned submission).

Input: `processed_train.csv`, `processed_test.csv` from `03_Preprocessing.ipynb`
(already imputed, encoded, and feature-selected)

Models compared: RandomForest, CatBoost, LightGBM (XGBoost optional — see note below)

**Competition metric: Balanced Accuracy** (mean of per-class recall) — NOT plain
accuracy. The target is imbalanced (class 1 ~86%, class 0 ~5.8%, class 2 ~8.4%),
so plain accuracy is misleadingly high; all CV tracking below uses
`balanced_accuracy_score` to match what Kaggle actually scores.
All 3 models also use class-balancing (`class_weight='balanced'` /
`auto_class_weights='Balanced'`) so they don't just favor the majority class.

The final, fully tuned submission (after `05_Feature_Engineering.ipynb`) belongs
in `06_Final_Submission.ipynb`, not here.

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
N_SPLITS = 5
CLASS_NAMES = ['fit','at-risk', "unhealthy"]

## 1. Load data

In [17]:
processed_train = pd.read_csv('../data/processed/processed_train.csv')
processed_test = pd.read_csv('../data/processed/processed_test.csv')

y = processed_train['health_condition']
X = processed_train.drop(columns=['health_condition'])
X_test = processed_test.copy()

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)
print('Class distribution:')
print(y.value_counts(normalize=True).round(3))

X shape: (690088, 16)
X_test shape: (295753, 16)
Class distribution:
health_condition
1    0.859
2    0.084
0    0.058
Name: proportion, dtype: float64


## 2. Define models to compare
Each entry is a factory function so a fresh, unfitted model is created for every fold
(reusing the same fitted model across folds would leak information between folds).

In [18]:
def make_random_forest():
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    )

def make_catboost():
    return CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=8,
        random_state=RANDOM_STATE,
        verbose=False,
        early_stopping_rounds=100,
        auto_class_weights='Balanced'
    )

def make_lightgbm():
    return LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    )

# XGBoost is optional — uncomment if you have it installed and want a 4th candidate
# from xgboost import XGBClassifier
# def make_xgboost():
#     return XGBClassifier(
#         n_estimators=1500, learning_rate=0.05, max_depth=8,
#         random_state=RANDOM_STATE, n_jobs=-1, eval_metric='mlogloss'
#     )

model_factories = {
    'RandomForest': make_random_forest,
    'CatBoost': make_catboost,
    'LightGBM': make_lightgbm,
    # 'XGBoost': make_xgboost,
}

## 3. Cross-validation comparison
StratifiedKFold keeps the class proportions consistent across folds — important here
since the target has 3 classes that may not be perfectly balanced.

CatBoost and LightGBM use `eval_set` + early stopping so `n_estimators`/`iterations`
above are just an upper bound, not a fixed number of rounds.

In [19]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_results = {name: [] for name in model_factories}
oof_preds = {name: np.zeros(len(X)) for name in model_factories}
oof_proba = {name: np.zeros((len(X), 3)) for name in model_factories}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    print(f'\n=== Fold {fold + 1}/{N_SPLITS} ===')

    for name, factory in model_factories.items():
        model = factory()

        if name == 'CatBoost':
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        elif name == 'LightGBM':
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
        else:
            model.fit(X_tr, y_tr)

        val_proba = model.predict_proba(X_val)
        val_pred = val_proba.argmax(axis=1)
        acc = balanced_accuracy_score(y_val, val_pred)

        cv_results[name].append(acc)
        oof_preds[name][val_idx] = val_pred
        oof_proba[name][val_idx] = val_proba

        print(f'  {name:<15} fold balanced_acc = {acc:.5f}')


=== Fold 1/5 ===
  RandomForest    fold balanced_acc = 0.90621
  CatBoost        fold balanced_acc = 0.90829


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007920 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
  LightGBM        fold balanced_acc = 0.90506

=== Fold 2/5 ===
  RandomForest    fold balanced_acc = 0.90637
  CatBoost        fold balanced_acc = 0.90980


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024477 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1806
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
  LightGBM        fold balanced_acc = 0.90568

=== Fold 3/5 ===
  RandomForest    fold balanced_acc = 0.90612
  CatBoost        fold balanced_acc = 0.90934


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004537 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
  LightGBM        fold balanced_acc = 0.90664

=== Fold 4/5 ===
  RandomForest    fold balanced_acc = 0.90458
  CatBoost        fold balanced_acc = 0.90807


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.058348 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 552071, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
  LightGBM        fold balanced_acc = 0.90371

=== Fold 5/5 ===
  RandomForest    fold balanced_acc = 0.90369
  CatBoost        fold balanced_acc = 0.90740


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015740 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 552071, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
  LightGBM        fold balanced_acc = 0.90342


## 4. Compare CV scores across models (balanced accuracy)

In [20]:
summary = pd.DataFrame({
    name: {
        'mean_bal_acc': np.mean(scores),
        'std_bal_acc': np.std(scores),
        'oof_bal_acc': balanced_accuracy_score(y, oof_preds[name])
    }
    for name, scores in cv_results.items()
}).T.sort_values('oof_bal_acc', ascending=False)
summary

,mean_bal_acc,std_bal_acc,oof_bal_acc
CatBoost,0.908580,0.000873,0.908580
RandomForest,0.905395,0.001067,0.905395
LightGBM,0.904903,0.001204,0.904903


In [21]:
best_model_name = summary.index[0]
print(f'Best model by OOF balanced accuracy: {best_model_name}')
print(classification_report(y, oof_preds[best_model_name]))

Best model by OOF balanced accuracy: CatBoost
              precision    recall  f1-score   support

           0       0.52      0.92      0.66     39803
           1       0.99      0.88      0.93    592561
           2       0.58      0.93      0.71     57724

    accuracy                           0.88    690088
   macro avg       0.69      0.91      0.77    690088
weighted avg       0.93      0.88      0.89    690088



In [22]:
cm = confusion_matrix(y, oof_preds[best_model_name])
pd.DataFrame(cm, index=[f'true_{c}' for c in CLASS_NAMES],
             columns=[f'pred_{c}' for c in CLASS_NAMES])

,pred_fit,pred_at-risk,pred_unhealthy
true_fit,36621,2990,192
true_at-risk,33919,519382,39260
true_unhealthy,147,3941,53636


## 5. Checkpoint submission
Refit the best model on the FULL training set (not just one fold), predict on test,
and export a submission — purely to sanity-check that the public leaderboard score
is in the same ballpark as the OOF balanced accuracy above. If it's way off, something
in the pipeline (leakage, train/test mismatch, or a metric misunderstanding) needs
investigating before moving on.

In [23]:
final_model = model_factories[best_model_name]()

if best_model_name == 'CatBoost':
    final_model.fit(X, y, verbose=False)
elif best_model_name == 'LightGBM':
    final_model.fit(X, y)
else:
    final_model.fit(X, y)

test_pred = np.asarray(final_model.predict(X_test)).flatten()

# reverse the target mapping from 03_Preprocessing (fit:0, at-risk:1, unhealthy:2)
inverse_mapping = {0: 'fit', 1: 'at-risk', 2: 'unhealthy'}
test_pred_labels = pd.Series(test_pred).map(inverse_mapping)

sample_submission = pd.read_csv('../data/sample_submission.csv')
checkpoint_submission = sample_submission.copy()
checkpoint_submission['health_condition'] = test_pred_labels.values

checkpoint_submission.to_csv(
    f'../submission/checkpoint_{best_model_name.lower()}.csv',
    index=False
)
checkpoint_submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## 6. Tune custom class weights
`'balanced'` uses a fixed formula (inverse class frequency). It may be
over- or under-correcting. Try a small grid of custom weights on the fastest
model (LightGBM) with a lighter 3-fold CV, to keep this quick to iterate on.

Weight order matches `CLASS_NAMES` = ['fit', 'at-risk', 'unhealthy'] → class 1
(at-risk, the majority) is kept at weight 1; only w0 (fit) and w2 (unhealthy)
are varied.

In [24]:
weight_grid = [
    {0: 1, 1: 1, 2: 1},
    {0: 2, 1: 1, 2: 2},
    {0: 3, 1: 1, 2: 2},
    {0: 2, 1: 1, 2: 3},
    {0: 4, 1: 1, 2: 3}
]
skf_quick = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
weight_results = []

for weights in weight_grid:
    fold_scores = []
    for tr_idx, val_idx in skf_quick.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = LGBMClassifier(
            n_estimators=800, learning_rate=0.05, max_depth=8,
            random_state=RANDOM_STATE, n_jobs=-1,
            class_weight=weights
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
        val_pred = model.predict(X_val)
        fold_scores.append(balanced_accuracy_score(y_val, val_pred))

    weight_results.append({
        'weights': weights,
        'mean_bal_acc': np.mean(fold_scores),
        'std_bal_acc': np.std(fold_scores)
    })
    print(f'{weights}  ->  {np.mean(fold_scores):.5f}')

weight_summary = pd.DataFrame(weight_results).sort_values('mean_bal_acc', ascending=False)
weight_summary


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004731 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 460058, number of used features: 16
[LightGBM] [Info] Start training from score -2.852888
[LightGBM] [Info] Start training from score -0.152363
[LightGBM] [Info] Start training from score -2.481162


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003742 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 460059, number of used features: 16
[LightGBM] [Info] Start training from score -2.852852
[LightGBM] [Info] Start training from score -0.152368
[LightGBM] [Info] Start training from score -2.481138


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003039 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 460059, number of used features: 16
[LightGBM] [Info] Start training from score -1.759477
[LightGBM] [Info] Start training from score -0.445287
[LightGBM] [Info] Start training from score -1.675445


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005768 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 460059, number of used features: 16
[LightGBM] [Info] Start training from score -1.759510
[LightGBM] [Info] Start training from score -0.445280
[LightGBM] [Info] Start training from score -1.675440
{0: 4, 1: 1, 2: 3}  ->  0.88331


,weights,mean_bal_acc,std_bal_acc
4,"{0: 4, 1: 1, 2: 3}",0.883311,0.001007
3,"{0: 2, 1: 1, 2: 3}",0.879587,0.001233
2,"{0: 3, 1: 1, 2: 2}",0.876062,0.001109
1,"{0: 2, 1: 1, 2: 2}",0.873772,0.001111
0,"{0: 1, 1: 1, 2: 1}",0.857816,0.001334


In [25]:
best_weights = weight_summary.iloc[0]['weights']
print('Best custom weights found:', best_weights)
# If this beats 'balanced' from section 4, use `best_weights` in place of
# class_weight='balanced' for LightGBM (and try the equivalent scaled version
# for CatBoost's class_weights / RandomForest's class_weight) going forward.

Best custom weights found: {0: 4, 1: 1, 2: 3}


## 7. Ensemble the 3 models (average predict_proba)
Uses the OOF probabilities already collected in section 3 — no retraining
needed to evaluate the ensemble's OOF score.

In [26]:
oof_proba_avg = np.mean([oof_proba[name] for name in model_factories], axis=0)
oof_pred_ensemble = oof_proba_avg.argmax(axis=1)

ensemble_bal_acc = balanced_accuracy_score(y, oof_pred_ensemble)
print(f'Ensemble OOF balanced accuracy: {ensemble_bal_acc:.5f}')
print(f'Best single model OOF balanced accuracy: {summary.iloc[0]["oof_bal_acc"]:.5f}')

print()
print(classification_report(y, oof_pred_ensemble, target_names=CLASS_NAMES))

Ensemble OOF balanced accuracy: 0.90826
Best single model OOF balanced accuracy: 0.90858

              precision    recall  f1-score   support

         fit       0.56      0.91      0.69     39803
     at-risk       0.99      0.90      0.94    592561
   unhealthy       0.62      0.92      0.74     57724

    accuracy                           0.90    690088
   macro avg       0.72      0.91      0.79    690088
weighted avg       0.93      0.90      0.91    690088



## 8. Ensemble checkpoint submission
Only export this if section 7 actually beats the best single model above —
otherwise stick with the single-model checkpoint from section 5.

In [27]:
# Refit all 3 models on the FULL training set, average their test predict_proba
test_probas = []
for name, factory in model_factories.items():
    model = factory()
    if name == 'CatBoost':
        model.fit(X, y, verbose=False)
    else:
        model.fit(X, y)
    test_probas.append(model.predict_proba(X_test))

test_proba_ensemble = np.mean(test_probas, axis=0)
test_pred_ensemble = test_proba_ensemble.argmax(axis=1)

ensemble_submission = sample_submission.copy()
ensemble_submission['health_condition'] = pd.Series(test_pred_ensemble).map(inverse_mapping).values
ensemble_submission.to_csv('../submission/checkpoint_ensemble.csv', index=False)
ensemble_submission.head()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007665 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 690088, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## Summary

- Compared RandomForest, CatBoost, LightGBM with 5-fold Stratified CV,
  scored on **balanced accuracy** (the actual competition metric) —
  plain accuracy would have looked inflated (0.965) vs. the real
  leaderboard score (0.855), a metric mismatch, not a bug
- All 3 models use class-balancing so minority classes (fit, unhealthy)
  aren't ignored
- Best single model: **CatBoost**, OOF balanced accuracy 0.90858,
  leaderboard 0.90586 — closely matched, pipeline is trustworthy
- Tried custom class weights (section 6) and a weighted/2-way ensemble
  (section 7) — **neither beat CatBoost alone**. All 3 models are
  tree-based on the same feature set, so their errors are correlated
  (they fail on the same borderline `at-risk` cases) — ensembling just
  dilutes the best model instead of correcting it
- **Decision: use CatBoost alone going forward.** Ensemble approach
  dropped — not worth the added complexity for this dataset

Next: `05_Feature_Engineering.ipynb`, using the confusion matrix above
(`at-risk` absorbing borderline `fit`/`unhealthy` cases) to target new
features, then `06_Final_Submission.ipynb` for the official submission.